CASO PRATICO: USARE LLaMA 3VIA HUGGING FACE PER ANALISI TESTUALE

Fino a poco fa, far girare un modello come Llama 3 da 8 miliardi di parametri, richiedeva hardware potente, oggi, grazie ad alcune tecniche, possiamo farlo girare su un computer portatile o una stanza gratuita di Google Colab.

- Quantizzazione a 4-bit: far girare miliardi di parametri su una GPU consumer
- Sentiment Analysis Zero-shot: classificazione senza addestramento specifico, sfruttando intelligenza nativa di llama, per classificare testi senza aver mai mostrato un esempio
- Batch Processing: massimizzare il throughput per analisi su larga scala. Inviare gruppi di testi simultaneamente per massimizzare la velocità.

Usare Llama via Hugging Face per analisi 'testuale' significa usare un modello della famiglia Meta Llama3, per leggere un testo e svolgere operazioni come riassunto, classificazione, estrazione di informazioni, sentiment, individuazione di temi o trasformazione in output strutturato.
Per analisi testuale è adatta la versione Instruct, non il modello base. Il modello Instruct è stato ottimizzato per eseguire istruzioni e dialogare, quindi è molto più adatto a richiesta del tipo 'classifica questa mail', 'estrai articoli e quantità' o 'riassumi questo testo'
TESTO -> prompt/istruzione -> LLaMa 3 Instruct -> analisi -> risposta testuale o strutturata

Ci sono due modi, molto diversi tra loro, di usare LLaMA3 via Hugging Face:
1) da remoto: python -> Hugging Face InferenceClient -> Inference Provider -> LLaMA 3 -> risposta
2) in locale: Python -> Transformers -> LLaMA 3 caricato sul pc -> risposta

Come far stare un gigante in una scatola piccola?
Democratizzazione dell'AI: La Quantizzazione
Eseguire Llama 3 su hardware limitato
I modelli moderni come Llama 3 hanno un problema di stazza (dimensioni proibitive) per le GPU standard. 
Llama 3 8 miliardi, se caricato normalmente, occupa circa 16/20 GB di VRAM, la maggior parte delle schede video che abbiamo a casa non supera i 12. 
La quantizzazione riduce la precisione dei pesi per abbattere il consumo di memoria VRAM.
Immagina di dover descrivere l'altezza di una persona, invece di usare 10 cifre decimali (che è una precisone non utile) usiamo solo i cm.
Utilizzare la libreria 'bitsandbytes' per caricare il modello in formato 4-bit, rendendo possibile l'esecuzione anche su schede con soli 8 o 21 GB di memoria. E la democratizzazione dell'ai, portiamo la potenza di un super computer sulla scrivania di ogni sviluppatore.

Ma cosa succede ai numeri quando li comprimiamo così tanto?
Meccanismi di Compressione
Dal formato FP16 all'INT4
Passare dal formato 16 a int4 è un salto enorme. Immagina di comprimere una foto ad alta risuluzione in un .jpeg leggero. Perderete qualche dettaglio, ma l'immagine rimane chiarissima.
Con il formato Normal Float 4 (NF4) non tagliamo i numeri a caso, usiamo un modello stocastico che prevede la distribuzione originale dei pesi.
La memoria occupata scende drasticamente.
Aggiungiamo poi la Double Quantization per guadagnare ancora più spazio. Quantizzare le costanti di quantizzazione per risparmiare ulteriori bit critici per la stabilità

Implementazione con Transformers
Operativamente con hugging face, tutto questo si riduere in un oggetto di configurazione chiamato BitsAndBytesConfig. Diciamo al sistema: carica in 4 bit (load_in_4bit=True), usa il tipo NF4 e carica in bfloat16. Sebbene i pesi siano in 4-bit (per risparmiare spazio), i calcoli avvengono in bfloat16 (precisione maggiore), per evitare errore di overflow durante la forward pass.
L'uso del device_map=auto distribuisce i layer del modello tra GPU e CPU in modo intelligente se la memoria video è insufficiente.

Logica metematica che permette di non perder l'intelligenza durante la compressione

La Matematica della Quantizzazione
Liner Quantization
Mappiamo un intervallo di valori continui in un insieme discreto di livelli per rappresentare i pesi con meno bit.
Immagina una scala, invece di poter stare in qualsiasi punto tra due piani, potete stare solo sui gradini. Il segreto sta nel posizionare i gradini dove si trovano i dati più importanti. 
Il processo calcola un fattore di scala e un punto di zero per preservare la dinamica originale della matrice.
Questa ottimizzazione garantisce che il cervello del modello non subisca danni permanenti, mantentendo le sue capacità quasi intatte.

Una volta caricato il gigante sulla nostra macchina LLama 3, passiamo all'azione

Sentiment Analysis Zero-shot
Sfruttare l'intelligenza nativa.
Llama 3 ha appreso così tanto linguaggio durante il training (ha letto tanto articoli, libri e commenti) da poter classificare testi senza aver mai visto esempi specifici del task.
Invece di addestrare un classificatore, useremo un prompt strutturato per guidare il modello verso una scelta discreta tra etichette predefinite.
Diremo al modello "analizza questa recensione e rispoindi solo con una parola positivo o negativo"
Stiamo accedendo alla saggezza che il modello ha acquisito durante mesi di pre-training.

Ma come possiamo essere sicuri che la scelta del modello sia qualla corretta?
Inference Strategica.
Dal linguaggio alla categoria.
L'inferenza strategica, non è solo leggere il testo generato, possiamo guardare sotto il cofano e guardare i logit. I logit sono punteggi grezzi che il modello assegna ad ogni possibile parola del vocabolario. 
Se il modello deve sceglire tra positivo e negativo, possiamo vedere la probabilità esatta che assegna alle due etichette. Questo ci permette di avere una misura della confidenza (se il modello assegna 50% positivo, possiamo stare sicuri, se assegna il 50% forse la recensione è ambigua)

Per ottenere risultati industriali dobbiamo controllare l'output
Impostiamo la temperatura a 0 o vicino allo zero. Questo perchè non vogliamo la creatività, vogliamo la risposta più probabile e costante.
Se il modello fatifa a capire il compito, possiamo aggiungere il Few-Shot Examples includendo 2 e 3 esempi nel prompt può stabilizzare l'output se il modello tende a divagare.
Inoltre, definire chiaramente il System Message permette di definire il ruolo del modello (es. sei un analista di testi), tutto questo migliora la precisione della categorizzazione.

Ma se analiziamo una recensione alla volta, i tempi di elaborazione sono troppo lenti. Possiamo scalare il processo.

Log Likelihood Classification
Il vero professionista non genera testo in libertà, ma usa la log likelihood, invece di lasciare che il modello scriva quello che vuole, poniamo delle domande chiuse.
Questo non permette al modelli di divagare., genera direttamente il risultato di cui abbiamo bisogno.  

La vera differenza in termini di velocità la fa il Batch Processing
Se hai 10.000 recensioni, inviarla una alla volta alla GPU è come compare un solo chicco di riso alla volta, è un enorme spreco di tempo.
Le GPU fanno migliaia di calcolo in parallelo.

Ma per mettere più testi in un unico pacchetto dobbiamo risolvere il problema che le frasi umane hanno lunghezze diverse. Per fare questo aggiungiamo il padding, aggiungendo dei poken vuoti alle frasi più corte per renderle lunghe quanto le freasi più lunghe.

Ci sono poi altri piccoli trucchi.
- Left vs Right Padding: per la generazione è preferibile il padding a sinistra affinchè i token reali siano alla fine della sequenza di input. Aggiungiamo gli zeri a sinistra affinchè i token reali siano alla fine della sequenza di input, pronte per essere continuate dal modello..
- Utilizzare  oggetti di "DataLoader" di Pytorch: Utilizzare oggetti DataLoader,per gestire il caricamento asincrono dei dati mentre la GPU elabora il batch precedente.
La Flash Attention è una tecnica avanzata che ottimizza l'accesso alla memoria durante il calcolo dell'attenzione, aumentando drasticamente la velocità, senza cambiare il modello.

L'efficienza massima si ottiene quando troviamo il punto di saturazione della GPU. Il passaggio da inferenza seriale a batch trasforma un LLM in uno strumento di analisi industriale capace di processare migliaia di record.

In [ ]:
import os
import time
from dotenv import load_dotenv
from llama_cpp import Llama # pip install llama-cpp-python


# SEZIONE 1: DEFINIZIONE DEI DATI E LOGICA DI FORMATTAZIONE
# In questa sezione prepariamo i dati di input e definiamo come devono essere presentati al modello.

# Lista di stringhe che rappresentano i dati grezzi da analizzare.
# Ogni elemento sara' processato individualmente dal modello di linguaggio.
recensioni = [
    "Un capolavoro assoluto, la regia è incredibile!",
    "Pessimo film, una noia mortale dall'inizio alla fine.",
    "Non saprei, alcune parti sono belle ma il finale è debole.",
    "Esperienza visiva discreta, ma la trama manca di spessore.",
    "Il miglior film dell'anno, attori straordinari.",
    "Delusione totale, soldi buttati."
]

def format_prompt(text):
    """
    Questa funzione implementa il 'Prompt Template' specifico per l'architettura Llama 3.
    Struttura il testo inserendo dei marcatori (tag) che il modello riconosce per distinguere
    le istruzioni di sistema dall'input dell'utente.
    """
    return (
        "<|start_header_id|>system<|end_header_id|>\n"
        "Sei un analista di sentiment. Rispondi esclusivamente con: 'Positive', 'Negative' o 'Neutral'.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"Testo: {text}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )

# SEZIONE 2: CUORE COMPUTAZIONALE E LOGICA DI BENCHMARK
# Questa funzione incapsula l'istanza del modello e la logica di misurazione delle prestazioni.

def esegui_analisi(descrizione, num_threads, batch_size):
    """
    Parametri:
    - descrizione: Etichetta del test per i log di output.
    - num_threads: Numero di processori (core) assegnati alla computazione.
    - batch_size: Quantita' di dati processati contemporaneamente.
    """
    print(f"\nAvvio test: {descrizione}")
    print(f"Configurazione: {num_threads} Thread(s), Batch Size {batch_size}")
    
    # Istanziazione della classe Llama.
    # Questo oggetto carica il modello pesantemente quantizzato (.gguf) in memoria RAM.
    # La configurazione n_threads e n_batch determina l'efficienza computazionale su CPU.
    llm = Llama.from_pretrained(
        repo_id="bartowski/Meta-Llama-3-8B-Instruct-GGUF",
        filename="Meta-Llama-3-8B-Instruct-Q4_K_M.gguf",
        n_ctx=512,           # Dimensione massima della finestra di contesto (token)
        n_threads=num_threads, # Parallelismo a livello di istruzioni CPU
        n_batch=batch_size,    # Dimensione del blocco di token per il prompt processing
        n_gpu_layers=0,      # Specifica l'uso esclusivo della CPU (0 layer su GPU)
        verbose=False        # Disattiva i log interni della libreria per pulizia output
    )
    
    risultati_test = []
    # Registrazione del tempo iniziale per il calcolo delle metriche temporali
    start_time = time.time()
    
    # Iterazione sulla lista delle recensioni
    for rec in recensioni:
        # Trasformazione del testo semplice in un prompt strutturato
        prompt = format_prompt(rec)
        
        # Chiamata al metodo __call__ dell'oggetto llm, che esegue l'inferenza.
        # temperature=0.1 garantisce risposte quasi deterministiche.
        # stop definisce i simboli che interrompono la generazione di testo.
        response = llm(
            prompt,
            max_tokens=10,
            temperature=0.1, 
            stop=["<|eot_id|>", "\n"]
        )
        
        # Estrazione della risposta testuale dal dizionario restituito dal modello
        sentiment = response["choices"][0]["text"].strip()
        risultati_test.append(sentiment)
    
    # Calcolo della durata complessiva dell'operazione
    end_time = time.time()
    durata = end_time - start_time
    # Calcolo del Throughput: numero di operazioni completate per unita' di tempo (secondi)
    throughput = len(recensioni) / durata
    
    return durata, throughput, risultati_test

# SEZIONE 3: ESECUZIONE DEI TEST COMPARATIVI
# Qui confrontiamo due diverse strategie di utilizzo delle risorse hardware.

# Scenario A: Simulazione di un carico di lavoro non ottimizzato (esecuzione seriale).
tempo_lento, tp_lento, _ = esegui_analisi(
    "Scenario A: Configurazione Seriale/Limitata", 
    num_threads=1, 
    batch_size=8
)

# Scenario B: Utilizzo ottimizzato del parallelismo hardware tramite thread multipli e batch piu' ampi.
tempo_veloce, tp_veloce, risultati_finali = esegui_analisi(
    "Scenario B: Configurazione Parallelizzata/Batch", 
    num_threads=4, 
    batch_size=512
)

# SEZIONE 4: PRESENTAZIONE DEI RISULTATI E ANALISI METRICHE
# Confronto finale tra i due scenari per validare l'efficacia delle ottimizzazioni.

print(f"\n--- CONFRONTO PRESTAZIONI ---")
print(f"Tempo Scenario A (Limitato):   {tempo_lento:.2f} secondi")
print(f"Tempo Scenario B (Ottimizzato): {tempo_veloce:.2f} secondi")

# Calcolo matematico della percentuale di miglioramento tra i due tempi di esecuzione.
miglioramento = ((tempo_lento - tempo_veloce) / tempo_lento) * 100
print(f"Incremento di velocità: +{miglioramento:.1f}%")

# Il throughput indica quanti campioni il sistema riesce a gestire ogni secondo.
print(f"Throughput finale: {tp_veloce:.2f} campioni al secondo")

print("\n--- ESEMPIO RISULTATI FINALI ---")
# Visualizzazione pratica della classificazione effettuata dal modello.
for rec, sent in zip(recensioni[:3], risultati_finali[:3]):
    print(f"Testo: {rec[:40]}... -> Sentiment: {sent}")


Avvio test: Scenario A: Configurazione Seriale/Limitata
Configurazione: 1 Thread(s), Batch Size 8


c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\EPICODE\Epicode_Python_AI_MachineLearning\.venv\Lib\site-packages\huggingface_hub\utils\_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
